In [1]:
import sys, os
import numpy as np
import pandas as pd
from scipy import stats
from collections import Counter
import json

current_dir = os.path.dirname(os.path.abspath('__file__'))
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import tol_colors as tc

In [2]:
import scienceplots
plt.style.use(['science', 'no-latex'])
plt.rcParams['text.latex.preamble'] = r'\usepackage[cm]{sfmath}\usepackage{amsmath}\centering'
plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.it'] = 'Helvetica:italic'
plt.rcParams["text.usetex"] = False


In [3]:
potato_duplicate_questions = [14, 83, 121]
models = [
    "gemma-2-9b-it",
    "gemma-3-12b-it",
    "Llama-3.1-8B-Instruct",
    "Mistral-7B-Instruct-v0.3",
    "Phi-3.5-mini-instruct",
]

model_rename = {
    "gemma-2-9b-it": "Gemma-2-9B",
    "gemma-3-12b-it": "Gemma-3-12B",
    "Llama-3.1-8B-Instruct": "Llama-3.1-8B",
    "Mistral-7B-Instruct-v0.3": "Mistral-7B",
    "Phi-3.5-mini-instruct": "Phi-3.5-3.8B",
}

datasets = {
    "hotpot_qa_final": "HotpotQA",
    "squad_v2_final": "SQuAD 2.0",
    "potato_final": "POTATO",
    "bioasq_final": "BioASQ",
}

num_samples_list = [5, 10, 25, 50, 75, 100]

#### Information gain per denary

In [4]:
p = "no_preprompt"

In [5]:
def information_gain(labels1, labels2):
    if labels1 is not None and len(labels1) > 0:
        item_frequencies1 = np.array(list(dict(Counter(labels1)).values()))/len(labels1)
        entropy1 = stats.entropy(item_frequencies1)
    else:
        entropy1 = 0

    item_frequencies2 = np.array(list(dict(Counter(labels2)).values()))/len(labels2)
    entropy2 = stats.entropy(item_frequencies2)

    return entropy2 - entropy1

In [6]:
information_gain_data = {}

for model in models:
    information_gain_data[model] = {}

    uncertainty_df = pd.read_csv(f"data/{p}/{model}/uncertainty.csv")
    for dataset in datasets:
        fname = f"data/{p}/{model}/{dataset}_results.json"
        with open(fname) as f:
            summary = json.load(f)

        x_vals = range(10, 100+1, 10)
        information_gain_agg_all_queries = np.zeros(len(x_vals))

        # only iterate over the ids in the spreadsheet
        question_ids = uncertainty_df[uncertainty_df["dataset"]==dataset]["id"].unique()

        num_queries_traversed = 0
        for question_id in question_ids:
            question_id = str(question_id)
            num_queries_traversed += 1
            
            information_gain_agg = []

            for n in x_vals:                
                lower = n-10
                information_gain_denary = information_gain(
                    summary[question_id]["cluster_ids"]["nli-batch"]["100"][:lower], 
                    summary[question_id]["cluster_ids"]["nli-batch"]["100"][:n]
                )

                information_gain_agg.append(information_gain_denary)
                
            information_gain_agg_all_queries += np.array(information_gain_agg)

        information_gain_agg_all_queries /= num_queries_traversed
        information_gain_data[model][dataset] = information_gain_agg_all_queries
        print(model, dataset, "Done")

gemma-2-9b-it hotpot_qa_final Done
gemma-2-9b-it squad_v2_final Done
gemma-2-9b-it potato_final Done
gemma-2-9b-it bioasq_final Done
gemma-3-12b-it hotpot_qa_final Done
gemma-3-12b-it squad_v2_final Done
gemma-3-12b-it potato_final Done
gemma-3-12b-it bioasq_final Done
Llama-3.1-8B-Instruct hotpot_qa_final Done
Llama-3.1-8B-Instruct squad_v2_final Done
Llama-3.1-8B-Instruct potato_final Done
Llama-3.1-8B-Instruct bioasq_final Done
Mistral-7B-Instruct-v0.3 hotpot_qa_final Done
Mistral-7B-Instruct-v0.3 squad_v2_final Done
Mistral-7B-Instruct-v0.3 potato_final Done
Mistral-7B-Instruct-v0.3 bioasq_final Done
Phi-3.5-mini-instruct hotpot_qa_final Done
Phi-3.5-mini-instruct squad_v2_final Done
Phi-3.5-mini-instruct potato_final Done
Phi-3.5-mini-instruct bioasq_final Done


In [ ]:
square = False

if square:
    fig = plt.figure(figsize=(10, 10))
    gs = gridspec.GridSpec(3, 3, figure=fig)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[0, 2])
    ax4 = fig.add_subplot(gs[1, 0])
    ax5 = fig.add_subplot(gs[1, 1])
else:
    fig = plt.figure(figsize=(25, 5))
    gs = gridspec.GridSpec(1, 5, figure=fig)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[0, 2])
    ax4 = fig.add_subplot(gs[0, 3])
    ax5 = fig.add_subplot(gs[0, 4])
    
axes = [ax1, ax2, ax3, ax4, ax5]

colors = tc.get_colorset('muted')
dataset_colors = {
    "HotpotQA": colors[0],
    "SQuAD 2.0": colors[1],
    "POTATO": colors[3],
    "BioASQ": colors[4]
}

large_fontsize = 35
medium_fontsize = 30
small_fontsize = 25

for model_idx in range(len(models)):
    model = models[model_idx]
    ax = axes[model_idx]
    
    for dataset in datasets:
        ax.plot(
            np.array(x_vals)/10,
            information_gain_data[model][dataset], 
            "o",
            label=datasets[dataset],
            color=dataset_colors[datasets[dataset]],
            linestyle=None,
            lw=3
        )

        sns.despine(top=True, right=True, left=False, bottom=False, ax=ax)
        ax.xaxis.set_minor_locator(plt.NullLocator())
        ax.yaxis.set_minor_locator(plt.NullLocator())
        ax.tick_params(axis="y", which="both", right=False)
        ax.tick_params(axis="x", which="both", top=False) 

    ax.set_title(model_rename[model], fontsize=large_fontsize)
    ax.set_xlabel("Denary", fontsize=large_fontsize)
    if model_idx == 0:
        ax.set_ylabel("Information gain", fontsize=large_fontsize)
    ax.set_ylim(0., 1.75)
    ax.set_xscale("log")
    ax.set_xticks([1, 2, 5, 10])
    ax.set_xticklabels([1, 2, 5, 10])
    ax.set_yticks([0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75])
    ax.tick_params(axis='both', which='both', labelsize=small_fontsize)


handles, labels = ax.get_legend_handles_labels()
handles = handles[1:]+[handles[0]]
labels = labels[1:]+[labels[0]]
# interleave for 2 column setup
if square:
    handles = [handles[i] for i in range(1, len(handles), 2)]+[handles[i] for i in range(0, len(handles), 2)]
    labels = [labels[i] for i in range(1, len(labels), 2)]+[labels[i] for i in range(0, len(labels), 2)]
fig.legend(
    handles, labels, fontsize=small_fontsize, loc='lower center', 
        bbox_to_anchor=(0.5, -0.2), ncol=5
)

plt.tight_layout()
plt.savefig('figures/information_gain.pdf')